# 01 · EDA — Exploración de datos

**Proyecto Alerta** — plataforma de alerta temprana de riesgo agroclimático (Colombia).

Este notebook hace un primer reconocimiento de las tablas producidas por el pipeline
(`ingest → features → risk`) y almacenadas en `data/alerta.duckdb`:

- Inventario de tablas y conteo de filas.
- Esquema de la tabla maestra `features_municipio_cultivo` (26 variables).
- Cobertura geográfica y temporal (municipios, cultivos, periodos).
- Distribución del **Índice de Riesgo Agroclimático (IRA)** y sus niveles.
- Calidad de datos: valores faltantes.

> Requiere haber corrido el pipeline. Si la BD no existe, ejecuta `make pipeline`.


In [ ]:
# Arranque: localizar la raiz del repo y la base de datos DuckDB
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd

from config import config

DB_PATH = ROOT / config.duckdb_path
DB_OK = DB_PATH.exists()

print("Repo :", ROOT)
print("BD   :", DB_PATH)
print("Estado:", "OK, base encontrada" if DB_OK else "NO existe -> corre `make pipeline` antes de este notebook")


def con():
    """Conexion DuckDB de solo lectura con extension espacial cargada."""
    if not DB_OK:
        raise FileNotFoundError(
            f"No existe {DB_PATH}. Ejecuta el pipeline:  python scripts/run.py ingest && features && risk"
        )
    c = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        c.execute("INSTALL spatial; LOAD spatial;")
    except Exception:
        pass
    return c


def q(sql: str) -> pd.DataFrame:
    """Ejecuta SQL y devuelve un DataFrame."""
    with con() as c:
        return c.execute(sql).df()


In [ ]:
# Librerias de visualizacion (matplotlib obligatoria, seaborn opcional)
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="viridis")
    HAS_SNS = True
except Exception:
    HAS_SNS = False

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13

# Colores por nivel de riesgo (coherentes con el frontend)
NIVEL_COLOR = {"Bajo": "#2e7d32", "Medio": "#f9a825", "Alto": "#ef6c00", "Critico": "#c62828", "Crítico": "#c62828"}
print("matplotlib listo | seaborn:", HAS_SNS)


## 1. Inventario de tablas

Todo el proyecto se integra a través de DuckDB. Las tablas siguen el convenio
`raw_*` (crudo) → `clean_*` (limpio) → `features_*` (variables) → `ira_resultados` (salida).


In [ ]:
tablas = q("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""")
tablas


In [ ]:
# Conteo de filas por tabla
filas = []
for t in tablas["table_name"]:
    try:
        n = q(f'SELECT COUNT(*) AS n FROM "{t}"')["n"].iloc[0]
        filas.append((t, int(n)))
    except Exception as e:
        filas.append((t, f"error: {e}"))

conteo = pd.DataFrame(filas, columns=["tabla", "filas"]).sort_values("tabla")
conteo


## 2. La tabla maestra `features_municipio_cultivo`

Clave: `(codigo_municipio, cultivo, periodo)`. Contiene las **26 variables**
(14 SPC climáticas + 6 SEP productivas + 6 SVE socioeconómicas) que alimentan el motor de riesgo.


In [ ]:
fmc = q("SELECT * FROM features_municipio_cultivo")
print("Filas:", len(fmc), "| Columnas:", fmc.shape[1])
fmc.head()


In [ ]:
# Esquema (tipos de dato) de la tabla maestra
q("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'features_municipio_cultivo'
    ORDER BY ordinal_position
""")


## 3. Cobertura geográfica y temporal


In [ ]:
resumen = q("""
    SELECT
        COUNT(DISTINCT codigo_municipio) AS municipios,
        COUNT(DISTINCT cultivo)          AS cultivos,
        COUNT(DISTINCT periodo)          AS periodos,
        MIN(periodo)                     AS periodo_min,
        MAX(periodo)                     AS periodo_max
    FROM features_municipio_cultivo
""")
resumen


In [ ]:
# Top 15 cultivos por numero de registros
top_cultivos = q("""
    SELECT cultivo, COUNT(*) AS registros,
           COUNT(DISTINCT codigo_municipio) AS municipios
    FROM features_municipio_cultivo
    GROUP BY cultivo
    ORDER BY registros DESC
    LIMIT 15
""")
top_cultivos


## 4. Distribución del IRA

`ira_resultados` es la salida final del motor de riesgo: combina el IRA
(`0.5·SPC + 0.3·SEP + 0.2·SVE`) con las salidas de ML (anomalías y predicción de rendimiento).


In [ ]:
# OJO: los nombres viven en municipios_geom (1 fila por municipio), NO en
# ira_resultados. Hay que unir por codigo_municipio (igual que hace la API).
ira = q("""
    SELECT r.*, g.nombre_municipio, g.nombre_departamento
    FROM ira_resultados r
    LEFT JOIN municipios_geom g USING (codigo_municipio)
""")
print("Filas:", len(ira))
ira[["codigo_municipio", "nombre_municipio", "nombre_departamento", "cultivo",
     "spc", "sep", "sve", "ira_score", "ira_nivel"]].head()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].hist(ira["ira_score"].dropna(), bins=40, color="#00897b", edgecolor="white")
ax[0].set_title("Distribucion del IRA score")
ax[0].set_xlabel("IRA (0-1)"); ax[0].set_ylabel("frecuencia")

niveles = ira["ira_nivel"].value_counts().reindex(["Bajo", "Medio", "Alto", "Crítico"]).dropna()
colores = [NIVEL_COLOR.get(n, "#607d8b") for n in niveles.index]
ax[1].bar(niveles.index, niveles.values, color=colores, edgecolor="white")
ax[1].set_title("Registros por nivel de riesgo")
ax[1].set_ylabel("registros")

plt.tight_layout(); plt.show()


## 5. Municipios con mayor riesgo (última foto)


In [ ]:
# Ultima foto por municipio y su mayor IRA (sobre el DataFrame ya enriquecido con nombres)
top_riesgo = (ira.sort_values("periodo")
              .groupby("codigo_municipio", as_index=False).tail(1)
              .sort_values("ira_score", ascending=False)
              [["nombre_municipio", "nombre_departamento", "cultivo", "periodo",
                "ira_score", "ira_nivel"]]
              .head(15).round({"ira_score": 3}))
top_riesgo


## 6. Calidad de datos — valores faltantes

Los `NULL` son esperables: la anomalía de precipitación necesita histórico, algunas
variables SVE (NBI) pueden quedar vacías si la fuente DANE estuvo indisponible.


In [ ]:
faltantes = fmc.isna().mean().mul(100).round(1).sort_values(ascending=False)
faltantes = faltantes[faltantes > 0]
print("Columnas con faltantes (% de filas):")
faltantes.to_frame("% NaN")


---
### Conclusiones del EDA

- El pipeline entrega una tabla maestra por `municipio × cultivo × periodo` con 26 variables.
- El IRA se distribuye principalmente en niveles Bajo/Medio, con una cola de municipios Alto/Crítico → esos son los objetivos de la alerta.
- Hay faltantes esperables en variables que dependen de histórico o de fuentes con disponibilidad irregular.

Siguiente paso → **`02_limpieza_transformacion.ipynb`**: limpieza, codificación y normalización.
